<a href="https://colab.research.google.com/github/agilesh08/AI-refund-manager-Reft.ai/blob/main/RAGOptimised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install groq python-dotenv chromadb numpy pandas sentence-transformers langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu rank-bm25 pypdf docx2txt unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 8.2 MB/s eta 0:00:00
   ━━━━

In [ ]:
import os
import json
import time
import logging
import datetime
from typing import List, Dict, Any

import chromadb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import CrossEncoder

from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    TokenTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    UnstructuredFileLoader,
)
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.utils.math import cosine_similarity

C:\Users\arisc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\arisc\AppData\Local\Temp\ipykernel_9228\3174656246.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma, FAISS


In [ ]:
dir = "handbook-master"

### Loading files with different formats ...

In [ ]:
def load_all_files(dir_path):
    result = []

    for file in os.listdir(dir_path):
        file_path = os.path.join(dir_path, file)
        ext = file.lower().split(".")[-1]

        try:
            # ----- PDF FILE -----
            if ext == "pdf":
                loader = PyPDFLoader(file_path)
                docs = loader.load_and_split()     # automatically creates page metadata
                result.extend(docs)

            # ----- DOCX FILE -----
            elif ext == "docx":
                loader = Docx2txtLoader(file_path)
                content = loader.load()[0].page_content
                result.append(
                    Document(
                        page_content=content,
                        metadata={"file_path": file_path, "page": 1}
                    )
                )

            # ----- DOC FILE (.doc) -----
            elif ext == "doc":
                # Unstructured loader internally uses LibreOffice if available
                loader = UnstructuredFileLoader(file_path)
                docs = loader.load()
                for d in docs:
                    d.metadata["file_path"] = file_path
                    d.metadata["page"] = 1
                result.extend(docs)

            # ----- MARKDOWN FILE (.md) -----
            elif ext == "md":
                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()
                result.append(
                    Document(
                        page_content=text,
                        metadata={"file_path": file_path, "page": 1}
                    )
                )

            # ----- TEXT FILE (.txt) -----
            elif ext == "txt":
                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()
                result.append(
                    Document(
                        page_content=text,
                        metadata={"file_path": file_path, "page": 1}
                    )
                )

            # ----- ANY OTHER FILE (fallback) -----
            else:
                loader = UnstructuredFileLoader(file_path)
                docs = loader.load()
                for d in docs:
                    d.metadata["file_path"] = file_path
                    d.metadata["page"] = 1
                result.extend(docs)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    return result

In [ ]:
result = load_all_files(dir)
result

[Document(metadata={'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1}, page_content="# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).\n\n### Medical Insurance\n\nIn the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be eligible for continued coverage after that (COBRA).\n\nEach pay period, you’ll see a payroll deduction for medical insurance:\n\n* Employee-only medical cover

### Chunking the data and testing multiple strategies

In [ ]:
def create_chunks( text: str, chunking_method: str, chunk_size: int, chunk_overlap: int, separator: str = '\n'):
    """
    Splits the text into equally distributed chunks with specific word overlap.
    Args:
        text (str): Text to be converted into chunks.
        chunk_length (int): Maximum number of words in each chunk.
        chunk_overlap (int): Number of words to overlap between chunks.
        Additional parameters can be passed using kwargs.
    """
    print(f"Selected text splitter: {chunking_method}")
    if chunking_method=="CharacterTextSplitter":
        text_splitter = CharacterTextSplitter(
                    separator=separator,
                    chunk_size=chunk_size,
                    chunk_overlap=chunk_overlap,
                    length_function=len,
                    add_start_index=True)
    elif chunking_method == "TokenTextSplitter":
        text_splitter = TokenTextSplitter(
                    chunk_size=chunk_size,
                    chunk_overlap=chunk_overlap,
                    length_function=len,
                    add_start_index=True)
    elif chunking_method=="RecursiveCharacterTextSplitter":
        text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size = chunk_size,
                    chunk_overlap  = chunk_overlap,
                    length_function=len,
                    add_start_index=True)
    chunks = text_splitter.split_documents(text)
    print("Documents Split into " + str(len(chunks)) + " chunks")
    return chunks

In [ ]:
chunks = create_chunks(result, chunking_method="RecursiveCharacterTextSplitter", chunk_size=500, chunk_overlap=50)

Selected text splitter: RecursiveCharacterTextSplitter
Documents Split into 311 chunks


In [ ]:
chunks

[Document(metadata={'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1, 'start_index': 0}, page_content='# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).\n\n### Medical Insurance'),
 Document(metadata={'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1, 'start_index': 239}, page_content='In the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be'),
 Document(metada

In [ ]:
chunks1 = create_chunks(result, chunking_method="RecursiveCharacterTextSplitter", chunk_size=1000, chunk_overlap=50)

Selected text splitter: RecursiveCharacterTextSplitter
Documents Split into 146 chunks


In [ ]:
chunks1

[Document(metadata={'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1, 'start_index': 0}, page_content='# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).\n\n### Medical Insurance\n\nIn the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be eligible for continued coverage after that (COBRA).\n\nEach pay period, you’ll see a payroll deduction for medical insurance:'),
 Document(m

In [ ]:
chunks2 = create_chunks(result, chunking_method="RecursiveCharacterTextSplitter", chunk_size=1500, chunk_overlap=50)

Selected text splitter: RecursiveCharacterTextSplitter
Documents Split into 99 chunks


In [ ]:
chunks2

[Document(metadata={'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1, 'start_index': 0}, page_content='# Benefits & Perks\n\n## Health Insurance\n\nDetailed information about all 37signals insurance policies and other benefits can be found in [Basecamp](https://3.basecamp.com/2914079/buckets/28168307/vaults/5060979274).\n\n### Medical Insurance\n\nIn the United States, medical insurance is provided through Blue Cross Blue Shield PPO. The company pays 75% of the premium and the employee pays the other 25%. Open enrollment is in November every year, with new coverage beginning December 1. Marriages and domestic partnerships are covered. You’re eligible for coverage on your first day of employment. If you are terminated or resign from 37signals, your coverage will end on the last day of the month of your separation date and you may be eligible for continued coverage after that (COBRA).\n\nEach pay period, you’ll see a payroll deduction for medical insurance:\n\n* Employee-

In [ ]:
def load_embedding_model(embedding_model_name: str):
    """
    This function loads a pre-trained text embedding model that can be used for various downstream  tasks.
    It provides a convenient interface for accessing the text embedding functionality.

    Args:
        embedding_model_name (str): Name of the text embedding model to be used.
                                    It specifies the pre-trained model that will be loaded or instantiated for text embedding task.
    Returns:
            The loaded text embedding model instance ready for use in downstream tasks.

    """
    embedding_model = SentenceTransformerEmbeddings(model_name=embedding_model_name, model_kwargs={"trust_remote_code":True})
    print(f"Loaded text embedding model: {embedding_model_name}")
    return embedding_model


In [ ]:

emb_model = load_embedding_model('sentence-transformers/all-MiniLM-L6-v2')

C:\Users\arisc\AppData\Local\Temp\ipykernel_9228\2852033577.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name=embedding_model_name, model_kwargs={"trust_remote_code":True})
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1610.82it/s]


Loaded text embedding model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
emb_model1 = load_embedding_model('intfloat/multilingual-e5-small')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1269.59it/s]


Loaded text embedding model: intfloat/multilingual-e5-small


In [ ]:
emb1 = emb_model.embed_query("Follow the rules")
emb2 = emb_model.embed_query("Cheating is all you need")

a1 = np.array(emb1)
a2 = np.array(emb2)

cosine_similarity(a1.reshape(1,-1),a2.reshape(1,-1))

array([[0.2927998]])

In [ ]:
emb1 = emb_model1.embed_query("Follow the rules")
emb2 = emb_model1.embed_query("Cheating is all you need")

a1 = np.array(emb1)
a2 = np.array(emb2)

cosine_similarity(a1.reshape(1,-1),a2.reshape(1,-1))

array([[0.84247693]])

In [ ]:
def build_vector_store(store:str, langchain_docs:list, persist_dir: str, collection_name: str,embedding_model):
    """Build a new Chroma DB vector store.
    Args:
        langchain_docs (list): List of LangChain Documents
        collection_name (str): Unique name to assign to vector store
        embedding_model_name (str): The embedding model to use
    """
    if store == "chroma":
        client = chromadb.PersistentClient(persist_dir)
        colls = [c.name for c in client.list_collections()]
        if collection_name in colls:
            raise Exception(f"Collection `{collection_name}` already exists")

        # embedding_model = self.textEmbObj.load_embedding_model(embedding_model_name)
        vectordb = Chroma.from_documents(documents=langchain_docs,
                                            collection_name=collection_name,
                                            embedding=embedding_model,
                                            persist_directory=persist_dir)
        logging.info(f'Built vector store with {vectordb._collection.count()} entries...')
    else:
        # embedding_model = self.textEmbObj.load_embedding_model(embedding_model_name)
        vectordb=FAISS.from_documents(chunks,embedding_model)
        vectordb.save_local(folder_path=persist_dir, index_name=collection_name)
    return vectordb

In [ ]:
vectordb = build_vector_store(store="chroma",langchain_docs= chunks, persist_dir = "dir3", collection_name = "collection",embedding_model = emb_model)
vectordb

In [ ]:
vectordb1 = build_vector_store(store="faiss",langchain_docs= chunks, persist_dir = "dir4", collection_name = "collection",embedding_model = emb_model)
vectordb1

In [ ]:
retrieved_docs=vectordb.similarity_search_with_score('Weeks spent in cooldown',k=10)
retrieved_docs1=vectordb1.similarity_search_with_score('Weeks spent in cooldown',k=10)

In [ ]:
retrieved_docs

[(Document(metadata={'file_path': 'handbook-master\\how-we-work.md', 'start_index': 1181, 'page': 1}, page_content='## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs, when everyone writes up what they’ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication'),
  0.9672097563743591),
 (Document(metadata={'page': 1, 'start_index': 2447, 'file_path': 'handbook-master\\how-we-work.md'}, page_content='* Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.'),
  1.0688583850860596),
 (Document(metadata={'start_index': 4837, 'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1}, page_content='## Paid Time Off'),
  1.2162144184112549),
 (Document(metadata={'start_index': 5955, '

In [ ]:
retrieved_docs1

[(Document(id='437e3612-c699-4764-a338-96411aeea9e1', metadata={'file_path': 'handbook-master\\how-we-work.md', 'page': 1, 'start_index': 1181}, page_content='## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs, when everyone writes up what they’ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication'),
  np.float32(0.9672097)),
 (Document(id='4cdde7cb-0b75-4371-b3fc-eb508d08ed2d', metadata={'file_path': 'handbook-master\\how-we-work.md', 'page': 1, 'start_index': 2447}, page_content='* Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.'),
  np.float32(1.0688581)),
 (Document(id='9689ab65-b0eb-4796-b1a4-85e3fd23c4ad', metadata={'file_path': 'handbook-master\\benefits-and-perks.md'

In [ ]:
# 1. Convert your existing Chroma vector store into a dense retriever
chroma_retriever = vectordb.as_retriever(search_kwargs={"k": 10}) # Fetch top 10 from Chroma

# 2. Create the sparse (BM25) retriever from the original documents
# Note: Ensure 'langchain_docs' is the list of Documents used to build 'vectordb'
bm25_chroma_retriever = BM25Retriever.from_documents(
    documents=chunks,
    k=10 # Fetch top 10 from BM25
)

# 3. Combine them using EnsembleRetriever
# weights: Assigns importance; [0.5, 0.5] gives equal weight to BM25 and Chroma.
hybrid_chroma_retriever = EnsembleRetriever(
    retrievers=[bm25_chroma_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)

# 4. Perform the Hybrid Search
query = 'Weeks spent in cooldown'
hybrid_chroma_results = hybrid_chroma_retriever.invoke(query)
hybrid_chroma_results

[Document(metadata={'file_path': 'handbook-master\\how-we-work.md', 'page': 1, 'start_index': 2447}, page_content='* Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.'),
 Document(metadata={'file_path': 'handbook-master\\how-we-work.md', 'page': 1, 'start_index': 2652}, page_content='* *Heartbeats* are required of every team, and they’re due on the first Friday of the cooldown period. Teams use their heartbeat to summarize and celebrate the work they completed during the previous cycle, and the work described in the cycle heartbeat should line up (more or less) with the work you scheduled in the cycle kickoff.'),
 Document(metadata={'start_index': 1181, 'page': 1, 'file_path': 'handbook-master\\how-we-work.md'}, page_content='## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs

In [ ]:
# 1. Convert your existing FAISS vector store into a dense retriever
faiss_retriever = vectordb1.as_retriever(search_kwargs={"k": 10}) # Fetch top 10 from FAISS

# 2. Create the sparse (BM25) retriever from the original documents
# Note: Ensure 'chunks' is the list of Documents used to build 'vectordb1'
bm25_faiss_retriever = BM25Retriever.from_documents(
    documents=chunks,
    k=10 # Fetch top 10 from BM25
)

# 3. Combine them using EnsembleRetriever
# weights: Assigns importance; [0.5, 0.5] gives equal weight to BM25 and FAISS.
hybrid_faiss_retriever = EnsembleRetriever(
    retrievers=[bm25_faiss_retriever, faiss_retriever],
    weights=[0.5, 0.5]
)

# 4. Perform the Hybrid Search
query = 'Weeks spent in cooldown'
hybrid_faiss_results = hybrid_faiss_retriever.invoke(query)
# hybrid_faiss_results

In [ ]:
retrieved_docs

[(Document(metadata={'file_path': 'handbook-master\\how-we-work.md', 'start_index': 1181, 'page': 1}, page_content='## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs, when everyone writes up what they’ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication'),
  0.9672097563743591),
 (Document(metadata={'page': 1, 'start_index': 2447, 'file_path': 'handbook-master\\how-we-work.md'}, page_content='* Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.'),
  1.0688583850860596),
 (Document(metadata={'start_index': 4837, 'file_path': 'handbook-master\\benefits-and-perks.md', 'page': 1}, page_content='## Paid Time Off'),
  1.2162144184112549),
 (Document(metadata={'start_index': 5955, '

In [ ]:
context = [i[0].page_content for i in retrieved_docs]
context

['## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs, when everyone writes up what they’ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication',
 '* Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.',
 '## Paid Time Off',
 '### Paid Sick Time\n\nWe don’t offer a bank of sick days, nor do we ask you to track your sick days. When you’re sick, please notify your manager as soon as you know you’ll be out, and for how long you expect to be out.\n\nIf you’ll be away from work due to illness or injury for more than 7 consecutive work days, you may be required to file a short-term disability claim.\n\n37signals does not pay out for unused sick time upon resignation or termination.',
 '

In [ ]:
class Prompt:

    def __init__(self,system_prompt,user_prompt):
        self.system_prompt = system_prompt
        self.user_prompt = user_prompt

    def get_prompt(self,prompt_struct,context,question):
        self.user_prompt = self.user_prompt.format(CONTEXT = context, QUESTION = question )
        return prompt_struct.format(SYSTEM_PROMPT = self.system_prompt,USER_PROMPT = self.user_prompt)

In [ ]:
SYSTEM_PROMPT = """You are a helpful, respectful and honest assistant.
Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature.
If you don't know the answer to a question, please don't share false information."""

USER_PROMPT = """Read the context and answer the question.
If it cannot be answered, only say: 'Unanswerable'.
Answer should be concise and professional.
Make sure response is not cut off, and do not give an empty response.
Guidelines for Answering:
1. Understand the Context
2. Base answers solely on the information within the given context; do not rely on external knowledge.
3. Craft responses in full sentences to enhance clarity.
4. Be Concise and Relevant. Avoid unnecessary elaboration.
5. Provide answers without personal opinions or interpretations.
6. Keep your response format consistent, adapting it to fit the nature of the question
7. Rely solely on the provided context. Do not introduce external information.
8. Only Respond in the language of the question. Ensure that the answer is provided in the same language as the question, unless otherwise specified. So therefore, if a question is given in Spanish, you have to answer in Spanish
#### START CONTEXT
Context:
{CONTEXT}
#### END CONTEXT
Question:
{QUESTION}
Answer: """

prompt_struct = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>
{USER_PROMPT}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
question= "What are the number of weeks spent in cooldown?"
p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
prompt = p_obj.get_prompt(prompt_struct,context,question)
print(prompt)


<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful, respectful and honest assistant.
Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature.
If you don't know the answer to a question, please don't share false information.<|eot_id|><|start_header_id|>user<|end_header_id|>
Read the context and answer the question.
If it cannot be answered, only say: 'Unanswerable'.
Answer should be concise and professional.
Make sure response is not cut off, and do not give an empty response.
Guidelines for Answering:
1. Understand the Context
2. Base answers solely on the information within the given context; do not rely on external knowledge.
3. Craft responses in full sentences to enhance clarity.
4. Be Concise and Relevant. Avoid unnecessary elaboration.
5. Provide answers without personal opinions or interpretations.
6. Keep your response 

In [ ]:
load_dotenv()
DEFAULT_GROQ_MODEL_ID = os.getenv("GROQ_MODEL_ID", "llama-3.1-8b-instant")

class LLM:
    """
    Wrapper class for interacting with Groq-hosted LLMs while keeping the
    notebook's original generate_response(prompt) interface unchanged.

    Required environment variable:
    - GROQ_API_KEY

    Optional environment variable:
    - GROQ_MODEL_ID, defaults to llama-3.1-8b-instant
    """

    def __init__(self, llm_params: Dict[str, Any] = None, model_id: str = DEFAULT_GROQ_MODEL_ID):
        load_dotenv()

        self.llm_params = llm_params or {}
        self.model_id = model_id or DEFAULT_GROQ_MODEL_ID

        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise EnvironmentError(
                "Missing required environment variable: GROQ_API_KEY. "
                "Create a Groq API key and add it to your .env file."
            )

        try:
            self.client = Groq(api_key=api_key)
        except Exception as e:
            raise RuntimeError(f"Failed to initialize Groq client: {e}")

    def _groq_generation_params(self) -> Dict[str, Any]:
        """Map the notebook's watsonx-style params to Groq chat-completion params."""
        params = {}

        max_tokens = self.llm_params.get("max_tokens", self.llm_params.get("max_new_tokens"))
        if max_tokens is not None:
            params["max_tokens"] = max_tokens

        temperature = self.llm_params.get("temperature")
        if temperature is None and self.llm_params.get("decoding_method") == "greedy":
            temperature = 0
        if temperature is not None:
            params["temperature"] = temperature

        if "top_p" in self.llm_params:
            params["top_p"] = self.llm_params["top_p"]

        stop = self.llm_params.get("stop", self.llm_params.get("stop_sequences"))
        if stop:
            params["stop"] = stop

        return params


    # Response Generation

    def generate_response(self, prompt: str) -> str:
        """
        Generate a response from Groq for a given prompt.

        Args:
            prompt (str): The text prompt to send to the model.

        Returns:
            str: The generated response text.
        """
        try:
            completion = self.client.chat.completions.create(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                **self._groq_generation_params(),
            )
            return completion.choices[0].message.content
        except Exception as e:
            raise RuntimeError(f"Error generating response from Groq model '{self.model_id}': {e}")

In [ ]:
llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':500,
            'repetition_penalty':1.1,
            "temperature": 0.2,
            "top_k":50,
            "top_p":1
            }
llm_obj = LLM(llm_params = llm_params,model_id = DEFAULT_GROQ_MODEL_ID)
response = llm_obj.generate_response(prompt)
response

'We spend two weeks cooling down in between each cycle.'

In [ ]:
llm_params = {
            'decoding_method':"sample",
            'min_new_tokens':1,
            'max_new_tokens':500,
            'repetition_penalty':1.1,
            "temperature": 0.2,
            "top_k":100,
            "top_p":1
            }
llm_obj = LLM(llm_params = llm_params,model_id = DEFAULT_GROQ_MODEL_ID)
response = llm_obj.generate_response(prompt)
response

'According to the context, in between each cycle, the company spends two weeks cooling down.'

In [ ]:
llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':500,
            'repetition_penalty':1.1,
            "temperature": 0.8,
            "top_k":50,
            "top_p":1
            }
llm_obj = LLM(llm_params = llm_params,model_id = DEFAULT_GROQ_MODEL_ID)
response = llm_obj.generate_response(prompt)
response

'The number of weeks spent in cooldown is two weeks.'

In [ ]:
llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':500,
            'repetition_penalty':1.1,
            "temperature": 0.2,
            "top_k":50,
            "top_p":1
            }
llm_obj = LLM(llm_params = llm_params,model_id = 'groq/compound-mini')
response = llm_obj.generate_response(prompt)
response

'The cooldown period lasts **two weeks**.'

In [ ]:
llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':500,
            'repetition_penalty':1.1,
            "temperature": 0,
            "top_k":50,
            "top_p":1
            }
llm_obj = LLM(llm_params = llm_params,model_id = 'openai/gpt-oss-20b')
response = llm_obj.generate_response(prompt)
response

'Two weeks.'

In [ ]:
for i in retrieved_docs[:3]:
    print("Chunk -- ",i[0].page_content)
    print("------------------------------------")
    print("File -- ",i[0].metadata["file_path"])
    print("------------------------------------")
    print("Page -- ",i[0].metadata["page"])
    print("------------------------------------")

Chunk --  ## Cooldown

In between each cycle, we spend two weeks cooling down. That’s when product teams deal with bugs, when everyone writes up what they’ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.

## Communication
------------------------------------
File --  handbook-master\how-we-work.md
------------------------------------
Page --  1
------------------------------------
Chunk --  * Every team submits a *Kickoff* for the upcoming cycle, and they’re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.
------------------------------------
File --  handbook-master\how-we-work.md
------------------------------------
Page --  1
------------------------------------
Chunk --  ## Paid Time Off
------------------------------------
File --  handbook-master\benefits-and-perks.md
------------------------------------
Page 

In [ ]:
class Reranker:
    """
    Implements a cross-encoder-based reranker to reorder retrieved documents
    based on their semantic relevance to a given question.

    Workflow:
    1. **Pair Construction**: Creates (question, document) pairs.
    2. **Scoring**: Uses a cross-encoder model to compute semantic similarity scores.
    3. **Ranking**: Sorts documents by descending relevance score.
    """

    def __init__(self, model_name: str = "BAAI/bge-reranker-large"):
        """
        Initialize the reranker model.

        Args:
            model_name (str, optional): Hugging Face model ID for reranking.
                Defaults to 'BAAI/bge-reranker-large'.
        """
        self.model_name = model_name
        try:
            self.reranker = CrossEncoder(self.model_name)
        except Exception as e:
            raise RuntimeError(f"Failed to load reranker model '{model_name}': {e}")

    # ------------------------------------------------------------------
    # Reranking Logic
    # ------------------------------------------------------------------
    def rerank(self, documents: List[str], question: str) -> List[str]:
        """
        Re-rank documents based on their semantic similarity to the question.

        Args:
            documents (List[str]): List of candidate document texts.
            question (str): The user query or question.

        Returns:
            List[str]: Documents sorted by descending relevance score.
        """
        if not documents:
            return []

        # Create (question, document) pairs for scoring
        pairs = [[question, doc] for doc in documents]

        try:
            scores = self.reranker.predict(pairs)
        except Exception as e:
            raise RuntimeError(f"Error during reranking: {e}")

        # Sort documents by descending score
        ranked_docs = [
            doc for _, doc in sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
        ]

        return ranked_docs

In [ ]:
docs= []
rank_obj = Reranker()

question= "What are the number of weeks spent in cooldown?"

retrieved_docs=vectordb.similarity_search_with_score(question,k=10)

context = [i[0].page_content for i in retrieved_docs]
for doc in retrieved_docs[:3]:
    d = {}
    d["chunk"] = doc[0].page_content
    d["page"] = doc[0].metadata["page"]
    d["file"] = doc[0].metadata["file_path"]
    docs.append(d)

reranked_context = rank_obj.rerank(context,question)

p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
prompt = p_obj.get_prompt(prompt_struct,reranked_context,question)
print("Response -- ",llm_obj.generate_response(prompt))
print("DOCS -- ")
pretty_json_output = json.dumps(docs, indent=2)
print(pretty_json_output)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1168.74it/s]


Response --  Two weeks.
DOCS -- 
[
  {
    "chunk": "## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That\u2019s when product teams deal with bugs, when everyone writes up what they\u2019ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "* Every team submits a *Kickoff* for the upcoming cycle, and they\u2019re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "37signals offers 20 days of vacation and personal days plus [11 local holidays](https://3.basecamp.com/2914079/buckets/28168307/documents/5061131347) every year. We ask that you [track your time off](https://3.basecamp.com/2914079/projects/14971171). Yo

In [ ]:
class HyDERetriever:
    """
    Implements the HyDE (Hypothetical Document Embeddings) retrieval technique.

    Workflow:
    1. **Hypothetical Answer Generation**: Use an LLM to generate a short, hypothetical answer
       to the user's query â€” providing semantic grounding.
    2. **Vector Search**: Combine the query and generated answer to perform
       a more contextually rich vector search.
    3. **Output**: Return the top matching document contents.

    This enhances retrieval accuracy by leveraging the LLM's prior knowledge.
    """

    def __init__(self):
        """Initialize HyDE retriever parameters and the LLM used for generating hypothetical answers."""
        self.default_prompt = (
            "You have to answer a given query. Please generate a concise and relevant answer "
            "under 200 words."
        )

        self.llm_id = DEFAULT_GROQ_MODEL_ID
        self.llm_params = {
            "decoding_method": "greedy",
            "max_new_tokens": 500,
            "min_new_tokens": 1,
        }

        self.llm_obj = LLM(self.llm_params, self.llm_id)

    # ------------------------------------------------------------------
    # Hypothetical Answer Generation
    # ------------------------------------------------------------------
    def _generate_hypothetical_answer(self, query: str) -> str:
        """
        Generate a hypothetical answer using the LLM to enrich query context.

        Args:
            query (str): The user's original question.

        Returns:
            str: The generated hypothetical answer.
        """
        prompt = f"{self.default_prompt}\nQuery: {query}\nAnswer:"
        try:
            hypothetical_answer = self.llm_obj.generate_response(prompt).strip()
        except Exception as e:
            raise RuntimeError(f"Failed to generate hypothetical answer: {e}")
        return hypothetical_answer

    # ------------------------------------------------------------------
    # HyDE Retrieval Orchestration
    # ------------------------------------------------------------------
    def retrieve(self, query: str, vector_db, n_retrieve: int = 5) -> list[str]:
        """
        Perform HyDE retrieval using a generated hypothetical answer.

        Steps:
        1. Generate a hypothetical answer from the LLM.
        2. Combine the original query with the hypothetical answer.
        3. Perform vector search to retrieve the top-matching documents.

        Args:
            query (str): The user's original question.
            vector_db: Vector database object supporting `similarity_search_with_score`.
            n_retrieve (int, optional): Number of documents to retrieve. Defaults to 5.

        Returns:
            list[str]: List of retrieved document contents ranked by similarity.
        """
        hypothetical_answer = self._generate_hypothetical_answer(query)
        combined_query = f"{query}\n{hypothetical_answer}"

        try:
            retrieved_docs = vector_db.similarity_search_with_score(combined_query, k=n_retrieve)
        except Exception as e:
            raise RuntimeError(f"Vector retrieval failed: {e}")
        docs=[]
        for doc in retrieved_docs[:3]:
            d = {}
            d["chunk"] = doc[0].page_content
            d["page"] = doc[0].metadata["page"]
            d["file"] = doc[0].metadata["file_path"]
            docs.append(d)

        return [doc[0].page_content for doc in retrieved_docs],docs



In [ ]:
hyde_obj  = HyDERetriever()
hyde_context,docs = hyde_obj.retrieve(query = question,vector_db = vectordb )
p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
prompt = p_obj.get_prompt(prompt_struct,hyde_context,question)
print("Response -- ",llm_obj.generate_response(prompt))
print("DOCS -- ")
pretty_json_output = json.dumps(docs, indent=2)
print(pretty_json_output)

Response --  Two weeks.
DOCS -- 
[
  {
    "chunk": "## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That\u2019s when product teams deal with bugs, when everyone writes up what they\u2019ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "* Every team submits a *Kickoff* for the upcoming cycle, and they\u2019re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "37signals offers 20 days of vacation and personal days plus [11 local holidays](https://3.basecamp.com/2914079/buckets/28168307/documents/5061131347) every year. We ask that you [track your time off](https://3.basecamp.com/2914079/projects/14971171). Yo

In [ ]:
class RAGFusion:
    """
    Implements the RAG Fusion process using multiple query generation
    and Reciprocal Rank Fusion (RRF) for improved document retrieval.

    Workflow:
    1. **Query Generation**: Generate multiple semantically similar queries from the original input using an LLM.
    2. **Vector Search**: Perform vector-based search for each generated query to fetch top-matching documents.
    3. **Reciprocal Rank Fusion (RRF)**: Re-rank retrieved documents based on their occurrence and ranking across queries.
    4. **Metadata Enrichment**: Combine reranked results with metadata (e.g., source, page).
    5. **Output**: Return the reranked and metadata-enriched document list.
    """

    def __init__(self):
        """Initialize model configuration and LLM instance."""
        self.llm_id = DEFAULT_GROQ_MODEL_ID
        self.llm_params = {
        "temperature": 0,
        "max_tokens": 800,
    }
        self.llm_obj = LLM(self.llm_params, self.llm_id)

    # ------------------------------------------------------------------
    # Query Generation
    # ------------------------------------------------------------------
    def _build_query_prompt(self, base_query: str, n_queries: int) -> str:
        """
        Build an instruction prompt for generating semantically similar queries.

        Args:
            base_query (str): The original user query.
            n_queries (int): Number of alternate queries to generate.

        Returns:
            str: The formatted prompt for the LLM.
        """
        return f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an assistant that generates {n_queries} unique, semantically equivalent search queries.

You MUST return ONLY a valid Python list.

The output will be directly parsed using Python eval().

Failure to return valid Python syntax is considered an incorrect response.

<|eot_id|><|start_header_id|>user<|end_header_id|>

Instructions:

- Each query should preserve the original intent and meaning.
- Output MUST be a valid Python list.
- Every query must be enclosed in double quotes.
- Begin with '['
- End with ']'
- Do NOT output markdown.
- Do NOT output explanations.
- Do NOT output code blocks.
- Do NOT output "Output:".
- Do NOT output any text before or after the list.
- Avoid duplicates.

Example:

Question:
"How do I integrate IBM Watson services into my application?"

Expected Output:

[
"What steps are involved in incorporating IBM Watson services into an application?",
"Can you provide guidance on integrating IBM Watson into an app?",
"How can I integrate IBM Watson functionalities into my application?"
]

Now generate {n_queries} alternate search queries.

Original Question:

"{base_query}"

<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    def generate_queries(self, base_query: str, n_queries: int) -> list[str]:
        """
        Generate multiple semantically similar queries from the original one using LLM.

        Args:
            base_query (str): The original user query.
            n_queries (int): Number of queries to generate.

        Returns:
            list[str]: List of generated query strings.
        """
        prompt = self._build_query_prompt(base_query, n_queries)
        response = self.llm_obj.generate_response(prompt)

        print("=" * 100)
        print(response)
        print("=" * 100)

        # Clean and evaluate list safely
        response = response.replace("Output:", "").strip()
        # response = response.replace("'","")
        try:
            generated_queries = eval(response)
        except Exception:
            raise ValueError("Failed to parse generated queries from LLM output.")
        return generated_queries

    # ------------------------------------------------------------------
    # Reciprocal Rank Fusion (RRF)
    # ------------------------------------------------------------------
    def _reciprocal_rank_fusion(self, results_dict: dict, k: int = 60) -> dict:
        """
        Apply Reciprocal Rank Fusion (RRF) to merge results from multiple queries.

        Args:
            results_dict (dict): {query: {doc_content: [source, score], ...}, ...}
            k (int, optional): RRF constant to smooth the rank contribution. Defaults to 60.

        Returns:
            dict: {doc_content: fused_score}
        """
        fused_scores = {}
        for query, doc_map in results_dict.items():
            sorted_docs = sorted(doc_map.items(), key=lambda x: x[1][2], reverse=True)
            for rank, (doc, _) in enumerate(sorted_docs):
                fused_scores[doc] = fused_scores.get(doc, 0) + 1 / (rank + k)
        return dict(sorted(fused_scores.items(), key=lambda x: x[1], reverse=True))

    # ------------------------------------------------------------------
    # Metadata Enrichment
    # ------------------------------------------------------------------
    def _merge_metadata(self, all_results: dict, fused_scores: dict) -> dict:
        """
        Attach metadata (e.g., source, page) to fused documents.

        Args:
            all_results (dict): {query: {doc_content: [source, page, score]}}
            fused_scores (dict): {doc_content: fused_score}

        Returns:
            dict: {doc_content: [source, page, score, fused_score]}
        """
        combined_metadata = {
            doc: meta for query_res in all_results.values() for doc, meta in query_res.items()
        }

        for doc, fused_score in fused_scores.items():
            if doc in combined_metadata:
                combined_metadata[doc].append(fused_score)
        return combined_metadata

    # ------------------------------------------------------------------
    # RAG Fusion Retrieval Orchestration
    # ------------------------------------------------------------------
    def run(self, query: str, vector_db, n_queries: int = 2, n_retrieve: int = 4) -> list[str]:
        """
        Execute the full RAG Fusion pipeline.

        Steps:
        1. Generate semantically similar queries.
        2. Perform vector search for each query.
        3. Apply Reciprocal Rank Fusion to merge results.
        4. Merge metadata and rerank.
        5. Return ordered document contents.

        Args:
            query (str): The user's original query.
            vector_db: The vector database object supporting `similarity_search_with_score`.
            n_queries (int, optional): Number of queries to generate. Defaults to 2.
            n_retrieve (int, optional): Documents to retrieve per query. Defaults to 4.

        Returns:
            list[str]: Ranked document contents after RAG Fusion.
        """
        all_results = {}

        # Generate expanded queries
        expanded_queries = [query] + self.generate_queries(query, n_queries)
        docs = []
        # Retrieve results for each generated query
        for q in expanded_queries:
            results = vector_db.similarity_search_with_score(q, n_retrieve)
            # print(results)
            all_results[q] = {
                r[0].page_content: [r[0].metadata["page"],r[0].metadata["file_path"], r[1]]
                for r in results
            }


        # Apply reciprocal rank fusion
        fused_scores = self._reciprocal_rank_fusion(all_results)

        # Attach metadata and reorder results
        reranked = self._merge_metadata(all_results, fused_scores)

        # for i in reranked:
        docs=[]
        for chunk in reranked:
            d = {}
            d["chunk"] = chunk
            d["page"] = reranked[chunk][0]
            d["file"] = reranked[chunk][1]
            docs.append(d)

        return list(reranked.keys()),docs

In [ ]:
RF_obj  = RAGFusion()
RF_context,docs = RF_obj.run(question, vectordb)
prompt = p_obj.get_prompt(prompt_struct,RF_context,question)
print("Response -- ",llm_obj.generate_response(prompt))
print("DOCS -- ")
pretty_json_output = json.dumps(docs, indent=2)
print(pretty_json_output)

["What is the duration of cooldown in terms of weeks?", "How many weeks is the cooldown period?"]
Response --  Two weeks.
DOCS -- 
[
  {
    "chunk": "## Cooldown\n\nIn between each cycle, we spend two weeks cooling down. That\u2019s when product teams deal with bugs, when everyone writes up what they\u2019ve worked on, and when teams decide what to tackle next. Sometimes big batch projects extend into cool down, but we try to avoid that.\n\n## Communication",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "* Every team submits a *Kickoff* for the upcoming cycle, and they\u2019re due the second Friday of the cooldown period. Teams use their kickoff to summarize the work they have scheduled for the upcoming cycle.",
    "page": 1,
    "file": "handbook-master\\how-we-work.md"
  },
  {
    "chunk": "37signals offers 20 days of vacation and personal days plus [11 local holidays](https://3.basecamp.com/2914079/buckets/28168307/documents/5061131347) every

In [ ]:
import random
from typing import Dict, Any, List

class SmartQueryRouter:
    """
    Component 4: Smart Query Router

    Responsibilities:
    1. **Intent Classification**: Uses LLM to categorize queries (e.g., Factual, Complex, Coding, Chat).
    2. **Routing Logic**: Maps categories to specific RAG strategies (e.g., Vector Search vs. RAG Fusion).
    3. **A/B Testing**: Implements a probabilistic split to test experimental strategies against the control.
    """

    def __init__(self, llm_obj=None, ab_test_rate: float = 0.2):
        """
        Args:
            llm_obj: Instance of the LLM class (wrapper for Llama/OpenAI etc).
            ab_test_rate (float): Percentage of traffic (0.0 to 1.0) to route to the experimental path.
        """
        self.llm_obj = llm_obj
        self.ab_test_rate = ab_test_rate

        # Define available strategies
        self.strategies = {
            "DIRECT_LLM": "Direct LLM response (No RAG)",
            "VECTOR_RAG": "Standard Vector Search RAG",
            "RAG_FUSION": "Multi-query RAG Fusion",
            "HYBRID_SEARCH": "Keyword + Vector Hybrid Search"
        }

    # ------------------------------------------------------------------
    # 1. Intent Classification
    # ------------------------------------------------------------------
    def _build_classification_prompt(self, query: str) -> str:
        """
        Constructs the classification prompt.
        """
        return f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert Query Router. Your task is to classify the user's query into exactly one of the following categories:

1. "FACTUAL": Simple, specific questions requiring precise retrieval (e.g., "What is the capital of France?", "Who is the CEO of X?").
2. "COMPLEX": Multi-faceted, reasoning-heavy, or ambiguous questions requiring multiple perspectives (e.g., "Compare the impact of X and Y", "How do I implement this system?").
3. "SUMMARIZATION": Requests to summarize broad topics or documents.
4. "CODING": Specific programming syntax or debugging questions.
5. "CHAT": General greetings or conversational inputs not requiring external data.

Instructions:
- Analyze the query complexity and intent.
- Return ONLY the category name.
- Do not add explanations or punctuation.

<|eot_id|><|start_header_id|>user<|end_header_id|>
Query: "{query}"
Category:
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

    def classify_query(self, query: str) -> str:
        """
        Classifies the query into a predefined category.
        """
        if not self.llm_obj:
            return "COMPLEX" # Fallback if no LLM provided

        prompt = self._build_classification_prompt(query)
        response = self.llm_obj.generate_response(prompt)

        # Clean output to ensure we just get the label
        category = response.strip().replace('"', '').replace("'", "").upper()

        valid_categories = ["FACTUAL", "COMPLEX", "SUMMARIZATION", "CODING", "CHAT"]

        if category not in valid_categories:
            # Default fallback for ambiguous responses
            return "COMPLEX"

        return category

    # ------------------------------------------------------------------
    # 2. & 3. Routing & A/B Testing
    # ------------------------------------------------------------------
    def determine_strategy(self, category: str) -> Dict[str, Any]:
        """
        Determines the execution strategy based on category and A/B testing logic.

        Returns:
            dict: The selected strategy configuration.
        """
        is_experimental_group = random.random() < self.ab_test_rate

        decision = {
            "category": category,
            "is_experiment": is_experimental_group,
            "strategy": "",
            "reasoning": ""
        }

        # --- A/B TESTING LOGIC ---
        # Hypothesis: RAG Fusion is better for complex queries, but expensive.
        # Experiment: Try using simple Vector RAG for 20% of 'COMPLEX' queries to see if it's "good enough".

        if category == "COMPLEX":
            if is_experimental_group:
                decision["strategy"] = "VECTOR_RAG" # Challenger (Experiment)
                decision["reasoning"] = "A/B Test: Force Simple Vector Search on Complex Query"
            else:
                decision["strategy"] = "RAG_FUSION" # Champion (Control)
                decision["reasoning"] = "Standard routing for complex reasoning"

        elif category == "FACTUAL":
            # For factual, we might test Hybrid search vs Vector search
            decision["strategy"] = "VECTOR_RAG"
            decision["reasoning"] = "High precision lookup required"

        elif category == "CODING":
            decision["strategy"] = "RAG_FUSION"
            decision["reasoning"] = "Coding queries often benefit from multiple retrieved snippets"

        elif category == "CHAT":
            decision["strategy"] = "DIRECT_LLM"
            decision["reasoning"] = "No context retrieval needed"

        else:
            decision["strategy"] = "VECTOR_RAG" # Default safe fallback

        return decision

    # ------------------------------------------------------------------
    # Main Execution
    # ------------------------------------------------------------------
    def run(self, query: str) -> Dict[str, Any]:
        """
        Main entry point for the Router.

        Args:
            query (str): User input.

        Returns:
            dict: routing decision containing target pipeline and metadata.
        """
        # 1. Classify
        category = self.classify_query(query)

        # 2. Route & A/B Test
        route_decision = self.determine_strategy(category)

        return route_decision

# Example Usage Logic
if __name__ == "__main__":
    # Mocking the LLM object for demonstration
    mock_llm_params = {"decoding_method": "greedy"}
    mock_llm = LLM(mock_llm_params, DEFAULT_GROQ_MODEL_ID)

    router = SmartQueryRouter(llm_obj=mock_llm, ab_test_rate=0.3)

    # Simulating queries
    test_queries = [
        "What are the number of weeks spent in cooldown?", # Might be classified as FACTUAL or CODING
        "What are the key financial, health, or lifestyle benefits and perks offered to employees?", # COMPLEX
        "Hello there", # CHAT
    ]

    print(f"{'QUERY':<100} | {'CATEGORY':<10} | {'STRATEGY':<15} | {'EXP?'}")
    print("-" * 150)

    for q in test_queries:
        # Force specific mocks for demonstration in this standalone script
        # In real usage, the LLM would actually predict these
        # if "install" in q: override_cat = "FACTUAL"
        # elif "difference" in q: override_cat = "COMPLEX"
        # else: override_cat = "CHAT"
        route_result = router.run(q)

        # We manually bypass classify_query internal logic here just to show the router logic
        # strictly for this print loop, effectively simulating LLM output
        decision = route_result

        print(f"{q:<100} | {decision['category']:<10} | {decision['strategy']:<15} | {decision['is_experiment']}")

QUERY                                                                                                | CATEGORY   | STRATEGY        | EXP?
------------------------------------------------------------------------------------------------------------------------------------------------------
What are the number of weeks spent in cooldown?                                                      | FACTUAL    | VECTOR_RAG      | False
What are the key financial, health, or lifestyle benefits and perks offered to employees?            | SUMMARIZATION | VECTOR_RAG      | False
Hello there                                                                                          | CHAT       | DIRECT_LLM      | True


In [ ]:
query = "What are the number of weeks spent in cooldown?"
route_result = router.run(query)
print("Agent suggesting the route to follow:")
pretty_json_output = json.dumps(route_result, indent=2)
print(pretty_json_output)

Agent suggesting the route to follow:
{
  "category": "FACTUAL",
  "is_experiment": false,
  "strategy": "VECTOR_RAG",
  "reasoning": "High precision lookup required"
}


In [ ]:
if route_result["strategy"] == "RAG_FUSION":
    print("\n==========================")
    print("ðŸš€ Routing Strategy: RAG Fusion")
    print(f"ðŸ“¦ Category: {route_result['category']}")
    print("==========================\n")

    RF_obj = RAGFusion()
    print("ðŸ”Ž Running Fusion Pipeline...")
    RF_context, _ = RF_obj.run(query, vectordb)

    print("ðŸ§  Generating prompt...")
    prompt = p_obj.get_prompt(prompt_struct, RF_context, query)

    print("ðŸ¤– Querying LLM...")
    response = llm_obj.generate_response(prompt)

    print("\n----- RESPONSE -----\n")
    print(response)
    print("\n--------------------\n")


elif route_result["strategy"] == "VECTOR_RAG":
    print("\n===============================")
    print("ðŸ“š Routing Strategy: Vector RAG")
    print("===============================\n")

    llm_obj = LLM(llm_params=llm_params, model_id=DEFAULT_GROQ_MODEL_ID)
    p_obj = Prompt(SYSTEM_PROMPT, USER_PROMPT)

    print("ðŸ” Searching documents...")
    retrieved_docs = vectordb.similarity_search_with_score(query, k=10)

    print("ðŸ“„ Extracting context...")
    context = [i[0].page_content for i in retrieved_docs]

    print("ðŸ§  Generating prompt...")
    prompt = p_obj.get_prompt(prompt_struct, context, query)

    print("ðŸ¤– Querying LLM...")
    response = llm_obj.generate_response(prompt)

    print("\n----- RESPONSE -----\n")
    print(response)
    print("\n--------------------\n")


elif route_result["strategy"] == "DIRECT_LLM":
    print("\n===============================")
    print("âš¡ Routing Strategy: Direct LLM")
    print("===============================\n")

    llm_obj = LLM(llm_params=llm_params, model_id=DEFAULT_GROQ_MODEL_ID)
    p_obj = Prompt(SYSTEM_PROMPT, USER_PROMPT)

    print("ðŸ§  Generating direct prompt...")
    prompt = p_obj.get_prompt(prompt_struct, [], query)

    print("ðŸ¤– Querying LLM...")
    response = llm_obj.generate_response(prompt)

    print("\n----- RESPONSE -----\n")
    print(response)
    print("\n--------------------\n")


ðŸ“š Routing Strategy: Vector RAG

ðŸ” Searching documents...
ðŸ“„ Extracting context...
ðŸ§  Generating prompt...
ðŸ¤– Querying LLM...

----- RESPONSE -----

According to the given context, in between each cycle, we spend two weeks cooling down.

--------------------



In [ ]:
class Eval:
    def __init__(self):
        self.llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':10,
            'repetition_penalty':1.1,
            # 'stop_sequences':["\n"],
            # "temperature": 0.2,
            # "top_k":50,
            # "top_p":1
            }
        self.llm_obj = LLM(llm_params = self.llm_params,model_id = DEFAULT_GROQ_MODEL_ID )


    def get_relevancy_scores(self, ques, retrieved_chunks, rag_response):

        query_str: str = f"Query: {ques}\nResponse: {rag_response}"
        context_str: str = "\n\n".join(retrieved_chunks)

        template = f"""Your task is to assess whether the given information is accurate and supported by the context provided. You must answer with either YES or NO.
        - Answer YES if the information is accurate and finds support within any part of the provided context, regardless of other unrelated details in the context.
        - Answer NO if the information is inaccurate or unsupported by the context.
        Avoid providing any additional explanations beyond your YES or NO answer.

        Example:
        Query and Response:
        Query: What is IBM?
        Response: IBM, or International Business Machines, is a company that designs and manufactures a family of computers and provides various technological innovations and services.

        Context:
        ['Business trends\n04/08/2025, 11:55 IBM - Wikipedia\nhttps://en.wikipedia.org/wiki/IBM 7/33',
    'revenue, and 67th largest overall company by revenue in the United States. IBM ranked No. 38 on\nthe 2020 Fortune 500 rankings of the largest United States corporations by total revenue.[123] In\n2014, IBM was accused of using "financial engineering" to hit its quarterly earnings targets rather\nthan investing for the longer term.[124][125][126]\nThe key trends of IBM are (as at the financial year ending December 31):[127][128]\nCorporate affairs\nBusiness trends\n04/08/2025, 11:55 IBM - Wikipedia',
    "IBM's own Global Services personal computer consulting and customer service division. The\nresulting merged business units then became known simply as IBM Personal Systems Group.[55] A\n1990sâ€“2000s\n04/08/2025, 11:55 IBM - Wikipedia\nhttps://en.wikipedia.org/wiki/IBM 5/33",
    'research; it sold its microcomputer division to\nLenovo in 2005. IBM continues to develop\nmainframes, and its supercomputers have\nconsistently ranked among the most powerful in the\nworld in the 21st century. In 2018, IBM along with 91\n04/08/2025, 11:55 IBM - Wikipedia\nhttps://en.wikipedia.org/wiki/IBM 1/33',
    'In 1991 IBM began spinning off its many divisions into autonomous subsidiaries (so-called "Baby\nBlues") in an attempt to make the company more manageable and to streamline IBM by having\nother investors finance those companies.[41][42] These included AdStar, dedicated to disk drives\nand other data storage products; IBM Application Business Systems, dedicated to mid-range\ncomputers; IBM Enterprise Systems, dedicated to mainframes; Pennant Systems, dedicated to',
    '2017. Retrieved April 9, 2024.\n129. "Board of Directors" (https://www.ibm.com/investor/governance/board-of-directors.html). IBM.\nMarch 9, 2020. Archived (https://web.archive.org/web/20200708010734/https://www.ibm.com/i\nnvestor/governance/board-of-directors.html) from the original on July 8, 2020. Retrieved\nMarch 11, 2020.\n130. "International Business Machines Corporation Common Stock (IBM) Institutional Holdings" (htt']

        Answer:
        YES

        Query and Response:
        {query_str}

        Context:
        {context_str}

        Answer:"""

        evaluate_response = self.llm_obj.generate_response(template)
        # print(evaluate_response)
        evaluation = 1 if 'YES' in evaluate_response.upper() else 0

        return evaluation

    def get_context_relevancy(self, question: str, retrieved_chunks: list[str]) -> int:
        """
        Evaluates if the retrieved context is relevant to the given question.
        Returns 1 if relevant (YES), else 0.
        """

        context_str = "\n\n".join(retrieved_chunks)

        template = f"""Your task is to determine if the provided context is relevant to the question.
        You must answer strictly with either YES or NO.

        - Answer YES if the context contains information that could help answer the question.
        - Answer NO if the context is unrelated or not helpful for the question.
        Avoid providing any explanation.

        Example:
        Question: What is IBM known for?
        Context: IBM develops and sells computer hardware, middleware, and software.
        Answer: YES

        Question: What is IBM known for?
        Context: The Eiffel Tower is located in Paris.
        Answer: NO

        Question: {question}

        Context:
        {context_str}

        Answer:"""

        response = self.llm_obj.generate_response(template)
        evaluation = 1 if 'YES' in response.upper() else 0

        return evaluation

    def get_answer_relevancy(self, question: str, rag_response: str) -> int:
        """
        Evaluates if the RAG-generated answer is relevant to the given question.
        Returns 1 if relevant (YES), else 0.
        """

        query_str = f"Question: {question}\nAnswer: {rag_response}"

        template = f"""Your task is to determine whether the given answer is relevant to the question.
        You must answer strictly with either YES or NO.

        - Answer YES if the answer directly or logically addresses the question.
        - Answer NO if the answer is off-topic, incomplete, or unrelated.
        Do not provide any explanation.

        Example:
        Question: What is IBM?
        Answer: IBM is a technology company that produces computer hardware and software.
        Answer: YES

        Question: What is IBM?
        Answer: Paris is the capital of France.
        Answer: NO

        {query_str}

        Answer:"""

        response = self.llm_obj.generate_response(template)
        evaluation = 1 if 'YES' in response.upper() else 0

        return evaluation



    def get_faithfulness_scores(self, retrieved_chunks, rag_response):

        context_str: str = "\n\n".join(retrieved_chunks)

        template: str = f"""Your task is to ascertain the factual accuracy of the provided information by comparing it against the given context. You are to answer with either YES or NO:
        - Answer YES if the information accurately aligns with or is supported by any aspect of the context, irrespective of other unrelated content.
        - Answer NO if the information does not align with the context or introduces facts not supported by the context, indicating a potential hallucination or factual inaccuracy.
        Avoid providing any additional explanations with your YES or NO response.

        Information: {rag_response}
        Context: {context_str}
        Answer:"""

        evaluate_response = self.llm_obj.generate_response(template)
        # print(evaluate_response)
        evaluation = 1 if 'YES' in evaluate_response.upper() else 0

        return evaluation

In [ ]:
qs = ["What are the key financial, health, or lifestyle benefits and perks offered to employees?",
 'What are the core behavioral expectations and guidelines outlined?',
 "What is Basecamp's policy regarding employees taking on outside work or a second job i.e. moonlighting?",
 "What rules and guidelines are in place for employees managing and using company-provided devices?",
 "What are the fundamental philosophies and specific organizational guidelines that define 'How We Work' at Basecamp?",
 "What is Basecamp's philosophy regarding the 'Manager of One'?",
 'How does Basecamp determine employee salaries?',
 "What is the company's policy on 'Summer Hours'?",
 'What is the sabbatical policy at Basecamp?',
 "What is the 'Library Rules' concept regarding office noise and communication?",
 'How much is the annual Continuing Education allowance?',
 'What is the Wellness Allowance and what does it cover?',
 'Does Basecamp offer a hardware refresh for employee computers?',
 'What is the policy regarding moonlighting or side projects?',
 'How often does the entire company meet in person?',
 'What is the Co-working Space stipend?',
 "What is Basecamp's stance on negotiating salaries?",
 'How does the Profit Sharing scheme work?',
 'What is the parental leave policy for primary caregivers?',
 "What is the 'Work Can Wait' philosophy regarding communication?",
 'How does Basecamp handle charitable matching?',
 "What is the 'Holiday Gift' allowance?",
 'How are travel expenses handled for company meetups?',
 "What is the policy on long work weeks or 'burning the midnight oil'?",
 "How does the company view 'titles' within the organization?"]

In [ ]:
RF_obj  = RAGFusion()
hyde_obj  = HyDERetriever()
rank_obj = Reranker()
eval_obj = Eval()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1710.25it/s]


In [ ]:
def get_docs(question):
    retrieved_docs=vectordb.similarity_search_with_score(question,k=10)
    context = [i[0].page_content for i in retrieved_docs]
    for doc in retrieved_docs[:3]:
        d = {}
        d["chunk"] = doc[0].page_content
        d["page"] = doc[0].metadata["page"]
        d["file"] = doc[0].metadata["file_path"]
        docs.append(d)
    return context,docs

In [ ]:
hyde = {"context_rel":[],"faithfulness":[],"answer_rel":[],"refer":[]}
RF={"context_rel":[],"faithfulness":[],"answer_rel":[],"refer":[]}
rerank = {"context_rel":[],"faithfulness":[],"answer_rel":[],"refer":[]}
for question in qs:
    RF_context,docs_rf = RF_obj.run(question, vectordb)
    hyde_context,docs_hyde = hyde_obj.retrieve(query = question,vector_db = vectordb )
    context,docs_rr = get_docs(question)
    reranked_context = rank_obj.rerank(context,question)

    p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
    prompt_hyde = p_obj.get_prompt(prompt_struct,hyde_context,question)
    p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
    prompt_RF = p_obj.get_prompt(prompt_struct,RF_context,question)
    p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
    prompt_RR = p_obj.get_prompt(prompt_struct,reranked_context,question)

    response_Hyde = llm_obj.generate_response(prompt_hyde)
    response_RF = llm_obj.generate_response(prompt_RF)
    response_RR = llm_obj.generate_response(prompt_RR)

    hyde["context_rel"].append(eval_obj.get_context_relevancy(question,hyde_context))
    hyde["faithfulness"].append(eval_obj.get_faithfulness_scores(hyde_context,response_Hyde))
    hyde["answer_rel"].append(eval_obj.get_answer_relevancy(question,response_Hyde))
    hyde["refer"].append({"question":question,"response":response_Hyde,"docs":docs_hyde})

    RF["context_rel"].append(eval_obj.get_context_relevancy(question,RF_context))
    RF["faithfulness"].append(eval_obj.get_faithfulness_scores(RF_context,response_RF))
    RF["answer_rel"].append(eval_obj.get_answer_relevancy(question,response_RF))
    RF["refer"].append({"question":question,"response":response_RF,"docs":docs_rf})

    rerank["context_rel"].append(eval_obj.get_context_relevancy(question,reranked_context))
    rerank["faithfulness"].append(eval_obj.get_faithfulness_scores(reranked_context,response_RR))
    rerank["answer_rel"].append(eval_obj.get_answer_relevancy(question,response_RR))
    rerank["refer"].append({"question":question,"response":response_RR,"docs":docs_rr})

["What are the main financial benefits provided to employees?", "[What are the primary health benefits offered to employees?"]
["What are the key behavioral expectations and rules outlined?", "What are the core behavioral expectations and protocols outlined?"]
["What are Basecamp's rules for employees taking on outside work or a second job?", "[Basecamp policy on moonlighting employees]"]
["What policies govern employee use of company-provided devices?", "[What are the regulations for employees managing company-provided devices?"]
["What are the core principles and company policies that shape the work culture at Basecamp?", "What are the key philosophies and organizational guidelines that define Basecamp's work environment?"]
["What is Basecamp's approach to the 'Manager of One' concept?", "How does Basecamp view the role of a 'Manager of One'?"]
["What is the process Basecamp uses to calculate employee salaries?", "How does Basecamp determine employee compensation?"]
["What are the co

In [ ]:
metrics_data = {}
def metric(out:dict):
    c_r = sum(out["context_rel"])/len(out["context_rel"])
    f = sum(out["faithfulness"])/len(out["faithfulness"])
    a = sum(out["answer_rel"])/len(out["answer_rel"])
    return {
            "Context Relevance": c_r,
            "Faithfulness": f,
            "Answer Relevance": a
        }

metrics_data["Hyde"] = metric(hyde)

metrics_data["RAG Fusion"] = metric(RF)

metrics_data["Reranking"] = metric(rerank)

In [ ]:
# Convert the nested dictionary to a Pandas DataFrame for easy analysis
df = pd.DataFrame(metrics_data).T # .T transposes the data so approaches are rows

# 2. Calculate the overall average score for each approach
df['Overall Average'] = df.mean(axis=1)

# Sort by overall average performance
df_sorted = df.sort_values(by='Overall Average', ascending=False)

print("Performance Summary Table:")
# Display the results, formatted for readability
print(df_sorted.round(2).to_markdown(numalign="left", stralign="left"))
print("\n" + "="*50 + "\n")

# 3. Identify the best approach for each metric
best_results = {}
for metric in df.columns[:-1]: # Exclude 'Overall Average'
    # idxmax() finds the index (approach name) of the maximum value in that column
    best_approach = df[metric].idxmax()
    best_value = df[metric].max()
    best_results[metric] = (best_approach, best_value)

# 4. Identify the best overall approach
overall_winner = df_sorted.index[0]
overall_score = df_sorted.iloc[0]['Overall Average']

# 5. Output the final comparison
print("ðŸ† Best Performing Approach by Metric:")
for metric, (approach, value) in best_results.items():
    print(f"- {metric}: {approach} (Score: {value:.2f})")

print("\nðŸ¥‡ Overall Winner (Based on Average Score):")
print(f"The best overall approach is {overall_winner} with an average score of {overall_score:.2f}.")

Performance Summary Table:
|            | Context Relevance   | Faithfulness   | Answer Relevance   | Overall Average   |
|:-----------|:--------------------|:---------------|:-------------------|:------------------|
| Reranking  | 1                   | 0.8            | 0.88               | 0.89              |
| RAG Fusion | 0.96                | 0.8            | 0.88               | 0.88              |
| Hyde       | 1                   | 0.84           | 0.8                | 0.88              |


ðŸ† Best Performing Approach by Metric:
- Context Relevance: Hyde (Score: 1.00)
- Faithfulness: Hyde (Score: 0.84)
- Answer Relevance: RAG Fusion (Score: 0.88)

ðŸ¥‡ Overall Winner (Based on Average Score):
The best overall approach is Reranking with an average score of 0.89.


In [ ]:
test_data = [
    {
        "question": "What is Basecamp's philosophy regarding the 'Manager of One'?",
        "answer": "A 'Manager of One' is someone who sets their own goals and executes them without needing heavy direction or daily oversight. It means the employee is capable of managing their own time, attention, and tasks.",
        "file_name": "how-we-work.md"
    },
    {
        "question": "How does Basecamp determine employee salaries?",
        "answer": "Basecamp pays everyone in the same role at the same level the same salary. They target the top 10% of the market rates for San Francisco, regardless of where the employee actually lives.",
        "file_name": "making-a-career.md"
    },
    {
        "question": "What is the company's policy on 'Summer Hours'?",
        "answer": "From May 1st through August 31st, Basecamp operates on a 4-day work week (32 hours). Employees can choose to take either Friday or Monday off.",
        "file_name": "time-off.md"
    },
    {
        "question": "What is the sabbatical policy at Basecamp?",
        "answer": "Every three years of employment, employees are eligible for a 30-day paid sabbatical.",
        "file_name": "time-off.md"
    },
    {
        "question": "What is the 'Library Rules' concept regarding office noise and communication?",
        "answer": "The office (and virtual spaces) should be treated like a library. Conversations should be kept at a whisper or moved to private rooms so that others can maintain focus and deep work.",
        "file_name": "code-of-conduct.md"
    },
    {
        "question": "How much is the annual Continuing Education allowance?",
        "answer": "Employees receive a $1,000 USD annual stipend to spend on conferences, books, classes, or other learning materials.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "What is the Wellness Allowance and what does it cover?",
        "answer": "Employees receive $100 per month to support a healthy lifestyle. This can be used for gym memberships, yoga classes, massages, or fitness apps.",
        "file_name": "severance.md"
    },
    {
        "question": "Does Basecamp offer a hardware refresh for employee computers?",
        "answer": "Yes, Basecamp buys a new Mac for employees every 3 years. The old computer is the employee's to keep.",
        "file_name": "our-internal-systems.md"
    },
    {
        "question": "What is the policy regarding moonlighting or side projects?",
        "answer": "Employees are allowed to work on side projects as long as they do not compete with Basecamp, do not use company resources, and do not interfere with the employeeâ€™s work performance.",
        "file_name": "moonlighting.md"
    },
    {
        "question": "How often does the entire company meet in person?",
        "answer": "The company holds all-hands meetups twice a year in Chicago.",
        "file_name": "our-rituals.md"
    },
    {
        "question": "What is the Co-working Space stipend?",
        "answer": "For remote employees who prefer not to work from home, Basecamp provides a stipend of up to $200/month to rent a desk at a co-working space.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "What is Basecamp's stance on negotiating salaries?",
        "answer": "Basecamp does not negotiate salaries. Everyone at the same skill level and role is paid the same public rate to ensure fairness and eliminate bias.",
        "file_name": "making-a-career.md'"
    },
    {
        "question": "How does the Profit Sharing scheme work?",
        "answer": "When the company does well, 25% of the growth in the remaining profit pot is distributed to employees.",
        "file_name": "compensation.md"
    },
    {
        "question": "What is the parental leave policy for primary caregivers?",
        "answer": "Primary caregivers are eligible for 16 weeks of 100% paid leave.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "What is the 'Work Can Wait' philosophy regarding communication?",
        "answer": "Basecamp believes there is no expectation of immediate responses. Unless it is a true emergency, communication should be asynchronous to protect attention.",
        "file_name": "titles-for-support.md"
    },
    {
        "question": "How does Basecamp handle charitable matching?",
        "answer": "Basecamp matches employee charitable donations dollar-for-dollar, up to $2,000 per year.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "What is the 'Holiday Gift' allowance?",
        "answer": "Employees receive a specific allowance (often $1,000) around the end of the year to buy gifts for themselves or their families.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "How are travel expenses handled for company meetups?",
        "answer": "Basecamp pays for all airfare, lodging, and meals during company meetups.",
        "file_name": "benefits-and-perks.md"
    },
    {
        "question": "What is the policy on long work weeks or 'burning the midnight oil'?",
        "answer": "Basecamp discourages working more than 40 hours a week. They believe that 40 hours is plenty of time to do great work if you are focused, and sustained overtime leads to burnout.",
        "file_name": "how-we-work.md"
    },
    {
        "question": "How does the company view 'titles' within the organization?",
        "answer": "Titles are kept simple (e.g., Junior, Senior, Lead, Principal) and are used primarily to reflect experience and pay grade, rather than hierarchy or authority over others.",
        "file_name": "making-a-career.md"
    }
]

In [ ]:
# for d in test_data:
#     question = d["question"]
#     RF_context,_ = RF_obj.run(question, vectordb)
#     p_obj = Prompt(SYSTEM_PROMPT,USER_PROMPT)
#     prompt_RF = p_obj.get_prompt(prompt_struct,RF_context,question)
#     response_RF = llm_obj.generate_response(prompt_RF)
#     print("Question - ",question)
#     print("Response - ",response_RF)
#     print("---------------------------------")
#     break

In [ ]:
def check_answer_completeness(generated_answer:str,golden_answer:str):

    PROMPT = f"""
    Role:

    You are an objective evaluator tasked with comparing a Generated Answer against a Golden Answer (the reference truth).

    Task:

    Determine if the Generated Answer is "complete."

    Completeness is defined as follows:

    The Generated Answer must contain all the key facts, steps, or essential information present in the Golden Answer.

    Differences in phrasing, style, or structure are acceptable as long as the meaning is preserved.

    The presence of additional correct information in the Generated Answer should not penalize the score.

    If the Generated Answer misses any critical information or key points that are found in the Golden Answer, it is considered incomplete.

    Inputs:

    Golden Answer:
    {golden_answer}

    Generated Answer:
    {generated_answer}

    Output Instruction

    Analyze the answers based on the criteria above.

    If the Generated Answer is complete: Respond with 1.

    If the Generated Answer is incomplete: Respond with 0.

    Output only the number (1 or 0). Do not provide explanations.

    Output:"""

    llm_params = {
            'decoding_method':"greedy",
            'min_new_tokens':1,
            'max_new_tokens':5,
            'repetition_penalty':1.1,
            # "temperature": 0.2,
            # "top_k":50,
            # "top_p":1
            }
    llm_obj = LLM(llm_params = llm_params,model_id = DEFAULT_GROQ_MODEL_ID)
    response = llm_obj.generate_response(PROMPT)
    return response.strip()

def calc_ans_completeness(retrieved_docs):
    total_complete = 0
    for i in range(len(test_data)):
        q = test_data[i]["question"]
        for j in range(len(retrieved_docs["refer"])):
            if retrieved_docs["refer"][j]["question"] == q :
                response = retrieved_docs["refer"][j]["response"]
                out = check_answer_completeness(generated_answer=response,golden_answer=test_data[i]["answer"])
                out = eval(out)
                if out == 1:
                    total_complete+=1
                break
    return total_complete/len(test_data)

def check_citation_accuracy(retrieved_docs) -> bool:
    total_checks = 0
    for i in range(len(test_data)):
        q = test_data[i]["question"]
        for j in range(len(retrieved_docs["refer"])):
            if retrieved_docs["refer"][j]["question"] == q :
                docs = retrieved_docs["refer"][j]["docs"]
                file_set = set()
                for d in docs:
                    file_set.add(os.path.basename(d["file"]).lower())
                if test_data[i]["file_name"].lower() in  file_set:
                    total_checks +=1
                break

    return total_checks/len(test_data)

def check_response_time_frustration(response_time_seconds: float) -> bool:
    """
    Checks if the response time is under the 2-second frustration threshold.

    Args:
        response_time_seconds: The measured response time.

    Returns:
        True if the time is <= 2.0 seconds, False otherwise.
    """
    return response_time_seconds <= 2.0

In [ ]:
print("-"*90)
print("Citation Accuracy for Hyde")
print(check_citation_accuracy(hyde))
print("-"*90)
print("Citation Accuracy for RAG Fusion")
print(check_citation_accuracy(RF))
print("-"*90)
print("Citation Accuracy for Reranking")
print(check_citation_accuracy(rerank))

------------------------------------------------------------------------------------------
Citation Accuracy for Hyde
0.7
------------------------------------------------------------------------------------------
Citation Accuracy for RAG Fusion
0.75
------------------------------------------------------------------------------------------
Citation Accuracy for Reranking
0.8
